# Lab 4 — Reading and Writing Data Files

**Course:** Python for AI & Data  
**Analyst:** *(your name)*  
**Date:** *(today's date)*

---

## The BeanScene Scenario

Sarah has exported BeanScene's menu data to a CSV file and stored her performance thresholds in a JSON config file. Your job is to load both files, run the menu analysis, and save the results back to disk.

Work from top to bottom. Run every cell as you go.


---
## Task 1 — Confirm Your Environment

Before opening any files, confirm:
1. Your working directory is the repo root (not inside `notebooks/`)
2. The data files exist at the expected paths

If the paths show `False`, close JupyterLab, `cd` to the repo root, and relaunch.


In [10]:
import os
from pathlib import Path


print("Working directory:", os.getcwd())


RAW_DIR = Path("data") / "raw"
PROCESSED_DIR = Path("data") / "processed"

menu_path = RAW_DIR / "beanscene_menu.csv"
config_path = RAW_DIR / "beanscene_config.json"


print("Menu CSV exists:", menu_path.exists())
print("Config JSON exists:", config_path.exists())


Working directory: C:\Users\evovo\lab-04-reading-and-writing-data-files
Menu CSV exists: True
Config JSON exists: True


---
## Task 2 — Load the Menu from CSV

Read `beanscene_menu.csv` into a list of dictionaries.

After loading:
- Convert `price` to `float` and `units_sold` to `int` for each row
- `name` and `category` should remain strings

Print the first item and check the types to confirm everything loaded correctly.

**Hint:** Use `csv.DictReader` — it reads each row as a dictionary using the header row as keys.


In [11]:
import csv

menu = []

with open(RAW_DIR / "beanscene_menu.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        menu.append({
            "name": row["name"],
            "category": row["category"],
            "price": float(row["price"]),
            "units_sold": int(row["units_sold"])
        })

print(menu[0])


print(type(menu[0]["name"]))
print(type(menu[0]["category"]))
print(type(menu[0]["price"]))
print(type(menu[0]["units_sold"]))


{'name': 'Espresso', 'category': 'Hot Drinks', 'price': 3.5, 'units_sold': 42}
<class 'str'>
<class 'str'>
<class 'float'>
<class 'int'>


---
## Task 3 — Load the Config from JSON

Read `beanscene_config.json` into a Python dictionary.

After loading, print:
- The café name
- The `high` performance threshold
- The `low_performer_threshold`

**Note:** Unlike CSV, JSON numbers load as actual Python numbers — no conversion needed.


In [7]:
print(config["cafe_name"])
print(config["performance_thresholds"]["high"])
print(config["low_performer_threshold"])


BeanScene
250.0
150.0


---
## Task 4 — Analyse the Menu

Using the loaded `menu` list and `config` dictionary:

1. Calculate revenue for each item (`price * units_sold`)
2. Classify each item as `"High"`, `"Medium"`, or `"Low"` using the thresholds from `config` — **not hardcoded numbers**
3. Store the results in a new list called `results` — each item should be a dict with keys: `name`, `revenue`, `tier`
4. Print the results in a readable format

**Tip:** You can reuse the `classify_item` logic from Lab 3 — either copy the function or rewrite it here.


In [8]:
def classify_item(revenue):
    if revenue >= config["performance_thresholds"]["high"]:
        return "High"
    elif revenue >= config["performance_thresholds"]["medium"]:
        return "Medium"
    else:
        return "Low"


results = []

for item in menu:
    revenue = item["price"] * item["units_sold"]
    tier = classify_item(revenue)
    results.append({
        "name": item["name"],
        "revenue": revenue,
        "tier": tier
    })


for r in results:
    print(f"{r['name']}: ${r['revenue']:.2f} - {r['tier']}")



Espresso: $147.00 - Low
Latte: $289.75 - High
Cappuccino: $171.00 - Medium
Cold Brew: $145.00 - Low
Flat White: $199.75 - Medium
Matcha Latte: $181.50 - Medium
Iced Americano: $220.00 - Medium
Chai Latte: $194.75 - Medium


---
## Task 5 — Save Results to CSV

Write your `results` list to `data/processed/menu_results.csv`.

The output file should have three columns: `name`, `revenue`, `tier`.

Make sure `data/processed/` exists before writing (create it if needed).


In [12]:
import csv
from pathlib import Path


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


output_path = PROCESSED_DIR / "menu_results.csv"

with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "revenue", "tier"])
    writer.writeheader()
    writer.writerows(results)

print("File written to:", output_path)

File written to: data\processed\menu_results.csv


---
## Task 6 — Save a Summary to JSON

Build a summary dictionary and write it to `data/processed/weekly_summary.json`.

Include at minimum:
- `cafe_name` (from config)
- `week` (from config: `analysis_week`)
- `total_revenue` (sum of all revenues)
- `items_analysed` (number of items)
- `best_seller` (name of the item with highest revenue)

Use `indent=2` when writing so the file is human-readable.


In [13]:
import json


summary = {
    "cafe_name": config["cafe_name"],
    "week": config["analysis_week"],
    "total_revenue": sum(r["revenue"] for r in results),
    "items_analysed": len(results),
    "best_seller": max(results, key=lambda r: r["revenue"])["name"]
}


output_path = PROCESSED_DIR / "weekly_summary.json"

with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)

print("File written to:", output_path)
print(json.dumps(summary, indent=2))



File written to: data\processed\weekly_summary.json
{
  "cafe_name": "BeanScene",
  "week": "2024-W04",
  "total_revenue": 1548.75,
  "items_analysed": 8,
  "best_seller": "Latte"
}


---
## Task 7 — Verify Your Outputs

Re-load both files you just wrote and print their contents to confirm they were saved correctly.

This is a professional habit: always verify your outputs — don't assume they're correct.


In [14]:
print("=== menu_results.csv ===")
with open(PROCESSED_DIR / "menu_results.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)


print("\n=== weekly_summary.json ===")
with open(PROCESSED_DIR / "weekly_summary.json", "r") as f:
    loaded_summary = json.load(f)
    print(json.dumps(loaded_summary, indent=2))


=== menu_results.csv ===
{'name': 'Espresso', 'revenue': '147.0', 'tier': 'Low'}
{'name': 'Latte', 'revenue': '289.75', 'tier': 'High'}
{'name': 'Cappuccino', 'revenue': '171.0', 'tier': 'Medium'}
{'name': 'Cold Brew', 'revenue': '145.0', 'tier': 'Low'}
{'name': 'Flat White', 'revenue': '199.75', 'tier': 'Medium'}
{'name': 'Matcha Latte', 'revenue': '181.5', 'tier': 'Medium'}
{'name': 'Iced Americano', 'revenue': '220.0', 'tier': 'Medium'}
{'name': 'Chai Latte', 'revenue': '194.75', 'tier': 'Medium'}

=== weekly_summary.json ===
{
  "cafe_name": "BeanScene",
  "week": "2024-W04",
  "total_revenue": 1548.75,
  "items_analysed": 8,
  "best_seller": "Latte"
}


---
## ⭐ Optional Advanced Tasks

### Optional 1 — Low Performers File
Write a separate CSV to `data/processed/low_performers.csv` containing only the items classified as `"Low"`. Then look up append mode (`"a"`) and add a final row with the count of low performers.

### Optional 2 — Config-Driven Thresholds
Confirm your analysis is fully config-driven: open `data/raw/beanscene_config.json`, change the `high` threshold from `250` to `200`, save the file, and re-run Task 4. Verify the tiers change without editing any Python code. Then change it back.

### Optional 3 — Category Summary
Group the menu items by `category` and write a file `data/processed/category_summary.json` showing total revenue per category. (Hint: use a dictionary to accumulate revenue per category as you loop.)


In [ ]:
# Optional tasks — work here
